# KBS 뉴스 수집 — 사이트 직접 / 기간 모드 (Colab용)

KBS 뉴스 사이트(`news.kbs.co.kr`) 내부 API(`/api/getNewsList`)를 직접 호출해 일자별 전체 기사를 수집한다. 응답 JSON에 제목/본문/발행시각/카테고리/기자명/이미지 URL 등이 모두 포함되어 있어 **URL 단계와 본문 단계가 한 노트북에 통합**된다 (LPOD/SBS 트랙과 패턴은 비슷하지만 출력 단계가 한 번에 끝남).

- 입력: `press_ranges` — 언론사(press, 보통 'KBS_direct') + 기간(start_date, end_date)
- 출력: `data/본문_bs4_{press}_{YYMMDD}_{YYMMDD}.csv` (link/pubdate/category/title/body)
- 보조 출력: 기간별 수집 로그 JSON, 중간 재개용 temp JSON
- 특징: API 직접 호출, 기간 단위 통합 CSV, 일자별 임시 체크포인트, 완료 파일 건너뛰기
- 셀렉터/필드: title=newsTitle, body=originNewsContents, pubdate=regDate, category=contentsName, link=`/news/view.do?ncd={newsCode}`
- 출력 파일명에 `본문_bs4_` prefix 사용 — BS4가 아닌 API 호출이지만 `본문_통합` 노트북과의 호환성을 위해 통일
- 트랙 B(네이버 경유, press='KBS')와 파일명 충돌 방지를 위해 press 이름에 `_direct` 접미사 사용

In [ ]:
# # Colab 환경 세팅 — requests, pandas 설치 (Selenium 불필요, BS4도 불필요 — JSON API)
# !pip install -q requests pandas

In [ ]:
# Google Drive 마운트 — 중간에 끊겨도 데이터 보존
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Drive 안의 프로젝트 폴더로 이동
# 통신3사 트리와 분리하기 위해 KBS/SBS 등 언론사 직접 수집은 news/ 하위에 보관
import os
PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'
os.chdir(PROJECT_DIR)
print(f'현재 작업 폴더: {os.getcwd()}')

In [ ]:
import json
import math
import os
import random
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

# 로컬/Colab 비교를 위해 User-Agent 고정
USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36'

# 담당 언론사와 수집 기간 지정 — 날짜 형식: 'YYYY.MM.DD'
# press 하나당 start_date ~ end_date 안의 일자를 한 통합 CSV로 묶어 저장
# _direct 접미사로 네이버 경유본 본문(`본문_bs4_KBS_*.csv`)과 출력 파일 분리
press_ranges = [
    {'press': 'KBS_direct', 'start_date': '2026.05.05', 'end_date': '2026.05.11'},
]


# 기간 단위 작업 목록 생성 — press_ranges 한 항목당 jobs 1개
# 내부 일자 순회는 collect_for_period에서 처리하므로 일자별 분할 jobs는 만들지 않음
def build_period_jobs(press_ranges):
    jobs = []
    for item in press_ranges:
        # 'YYYY.MM.DD' 문자열을 datetime으로 파싱 — 기간 유효성 검사용
        start = datetime.strptime(item['start_date'], '%Y.%m.%d')
        end = datetime.strptime(item['end_date'], '%Y.%m.%d')
        # 시작 일자가 끝 일자보다 늦으면 작업 범위가 잘못된 것이므로 즉시 중단
        if start > end:
            raise ValueError(f"시작 일자가 끝 일자보다 늦습니다: {item}")
        jobs.append({
            'press': item['press'],
            'start_date': item['start_date'],
            'end_date': item['end_date'],
        })
    return jobs


# 변수 정의 (날짜 형식: 'YYYY.MM.DD')
# 생성된 jobs는 다음 셀에서 순서대로 실행
jobs = build_period_jobs(press_ranges)

print(f'총 작업 수: {len(jobs)}')
for job in jobs:
    print(job)

# 셀 3을 건너뛰고 실행해도 기본 프로젝트 경로를 사용할 수 있게 보완
try:
    PROJECT_DIR
except NameError:
    PROJECT_DIR = '/content/drive/MyDrive/Text-data-Analysis_26-Spring/news'

# 저장할 폴더 지정 — 본문 CSV, 임시 체크포인트, 수집 로그, 실패 목록이 모두 이 폴더에 저장
SAVE_DIR = Path(PROJECT_DIR) / 'notebook' / 'crawling' / 'data'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'저장 위치: {SAVE_DIR}')

# requests 세션 생성 — User-Agent / 언어 / Referer 헤더를 반복 요청에 일관 적용
# Referer를 KBS 카테고리 페이지로 두면 일반 브라우저 요청과 동일하게 보임
session = requests.Session()
session.headers.update({
    'User-Agent': USER_AGENT,
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
    'Referer': 'https://news.kbs.co.kr/news/pc/category/category.do?ref=pSiteMap',
})
print(f'User-Agent: {USER_AGENT}')

In [ ]:
# 서버 부담을 줄이기 위해 페이지/일자/job 사이에 랜덤 대기
# PAGE_PAUSE: 같은 일자 내 페이지 이동 (짧게)
# DAY_PAUSE: 같은 기간 내 일자 전환 (중간) — 통신3사 트랙과 동일 값
# JOB_PAUSE: 다른 언론사로 전환 (길게)
PAGE_PAUSE_RANGE_SEC = (0.4, 1.2)
DAY_PAUSE_RANGE_SEC = (2, 5)
JOB_PAUSE_RANGE_SEC = (5, 12)
REQUEST_TIMEOUT_SEC = 15  # API 응답 대기 상한 — 네트워크 멈춤 시 무한 hang 방지
SKIP_COMPLETED = True
ROWS_PER_PAGE = 100  # API 한 번에 받을 기사 수 — KBS가 페이지당 최대 한계가 있을 수 있어 보수적으로 100

# KBS 내부 API 엔드포인트 — 카테고리 페이지(/news/pc/category/category.do)가 비동기로 호출하는 것과 동일
API_NEWS_LIST_URL = 'https://news.kbs.co.kr/api/getNewsList'
API_NEWS_COUNT_URL = 'https://news.kbs.co.kr/api/getNewsListCount'


# 랜덤 대기 후 로그 출력
def polite_sleep(label, pause_range):
    pause_sec = random.uniform(*pause_range)
    print(f"{label} {pause_sec:.1f}초 대기")
    time.sleep(pause_sec)


# 파일명에 사용할 YYMMDD_YYMMDD 형식 기간 문자열 생성 (통신3사 트랙과 동일 시그니처)
# 예: 2026.05.01 ~ 2026.05.07 -> '260501_260507'
def make_period_suffix(start_date, end_date):
    return f"{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}"


# 본문 안의 줄바꿈/연속 공백을 하나의 공백으로 정리 (CSV/분석 단계 일관성 위해)
def normalize_body_text(text):
    return re.sub(r'\s+', ' ', text).strip()


# 일자 하나의 API 파라미터 생성 — datetimeBegin/End는 'YYYYMMDDHHmmss' 14자리
def build_api_param(date_ymd, page=1, rows=ROWS_PER_PAGE):
    return {
        'currentPageNo': page,
        'rowsPerPage': rows,
        'exceptPhotoYn': 'Y',  # 카테고리 페이지 기본값 그대로 — 사진뉴스는 제외
        'datetimeBegin': f'{date_ymd}000000',
        'datetimeEnd': f'{date_ymd}235959',
        'contentsCode': 'ALL',  # ctcd='0000'(전체)에 해당하는 API 값
        'localCode': '00',
    }


# 일자별 총 건수 조회 — 페이지 수 계산용
def fetch_total_count(date_ymd):
    response = session.get(API_NEWS_COUNT_URL, params=build_api_param(date_ymd, page=1, rows=1), timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    j = response.json()
    if not j.get('success'):
        raise RuntimeError(f"getNewsListCount 응답 실패: {j.get('message')}")
    return int(j.get('data', 0))


# 일자별 1 페이지 받기 — API 응답의 'data' 리스트 반환
def fetch_newslist_page(date_ymd, page):
    response = session.get(API_NEWS_LIST_URL, params=build_api_param(date_ymd, page=page), timeout=REQUEST_TIMEOUT_SEC)
    response.raise_for_status()
    j = response.json()
    if not j.get('success'):
        raise RuntimeError(f"getNewsList 응답 실패 (page {page}): {j.get('message')}")
    return j.get('data', [])


# 한 일자의 모든 페이지를 순회하며 기사 dict 리스트 반환 + 페이지별 통계
def collect_articles_for_day(date_ymd):
    # 총 건수로 페이지 수 결정 — 일자별 1회만 count 호출
    total = fetch_total_count(date_ymd)
    max_page = max(1, math.ceil(total / ROWS_PER_PAGE))

    all_articles = []
    page_stats = []

    # 1 ~ max_page 순회하며 누적 (페이지 사이 짧은 랜덤 대기)
    for page in range(1, max_page + 1):
        if page > 1:
            polite_sleep(f"  page {page} 받기 전", PAGE_PAUSE_RANGE_SEC)
        articles = fetch_newslist_page(date_ymd, page)
        all_articles.extend(articles)
        # 페이지별 수집량 — total 와 어긋나면 누락 의심
        page_stats.append({
            'page': page,
            'received': len(articles),
            'total_so_far': len(all_articles),
        })

    return all_articles, total, max_page, page_stats


# KBS API 응답 1건을 우리 표준 CSV 행으로 변환 (link/pubdate/category/title/body)
# title/body/pubdate 중 하나라도 비면 ValueError — 호출부에서 err_idx로 기록
def article_to_row(article):
    # 제목 — newsTitle (API 응답에 이미 정제된 형태)
    title = (article.get('newsTitle') or '').strip()

    # 본문 — originNewsContents 사용 (HTML 태그 없는 원본). newsContents는 <br /> 포함이라 비선호
    body_raw = article.get('originNewsContents') or article.get('newsContents') or ''
    body = normalize_body_text(body_raw)

    # 발행 시각 — regDate (입력 시각). serviceTime/deskTime과 거의 동일하지만 regDate가 원본 등록
    # modDate는 수정 시각이라 별도 보존 가능하지만 기본 pubdate는 입력 기준
    pubdate = (article.get('regDate') or '').strip()

    # 카테고리 — contentsName ('국제', '정치' 등 한글 라벨). contentsCode는 숫자라 비선호
    category = (article.get('contentsName') or '').strip()

    # 기사 URL 재조립 — 화면용 view.do 경로 + newsCode
    news_code = article.get('newsCode')
    if not news_code:
        raise ValueError('newsCode 없음 — API 응답 비정상')
    link = f'https://news.kbs.co.kr/news/view.do?ncd={news_code}'

    # 제목/본문/날짜 중 하나라도 없으면 실패 — 호출부에서 err_idx로 기록
    if not title or not body or not pubdate:
        raise ValueError(f'필드 누락: title={bool(title)}, body={bool(body)}, pubdate={bool(pubdate)}')

    return {
        'link': link,
        'pubdate': pubdate,
        'category': category,
        'title': title,
        'body': body,
    }


# 한 언론사의 기간 전체를 통합 CSV 1개로 저장
# 일자별 임시 체크포인트로 중단/재개 지원 (LPOD/SBS URL 노트북과 동일 패턴)
def collect_for_period(press, start_date, end_date, save_dir=SAVE_DIR):
    # 파일명 키로 쓸 기간 접미사 (예: 260501_260507)
    period = make_period_suffix(start_date, end_date)

    # 파일 경로 — temp: 일자 단위 체크포인트 / csv: 최종 통합 / stats: 수집 로그
    temp_path = save_dir / f"{press}_{start_date}_{end_date}_temp.json"
    csv_save_path = save_dir / f"본문_bs4_{press}_{period}.csv"
    stats_save_path = save_dir / f"수집로그_{press}_{period}.json"

    # 최종 CSV가 이미 있으면 같은 기간은 건너뜀 (재실행 시 idempotent)
    if SKIP_COMPLETED and csv_save_path.exists():
        print()
        print(f"=== {press} / {start_date} ~ {end_date} 이미 완료됨, 건너뜀 ===")
        print(f"기존 파일: {csv_save_path}")
        return csv_save_path

    print()
    print(f"=== {press} / {start_date} ~ {end_date} 수집 시작 ===")

    # 임시 파일에 기존 데이터가 있으면 불러오기 — last_date 다음 날부터 이어서 수집
    if temp_path.exists():
        with temp_path.open('r', encoding='utf-8') as f:
            checkpoint = json.load(f)
        all_rows = checkpoint.get('rows', [])
        err_records = checkpoint.get('err_records', [])
        last_collected_date = checkpoint.get('last_date')
        daily_stats = checkpoint.get('daily_stats', [])
        print(f"기존 임시 파일에서 행 {len(all_rows)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
    else:
        all_rows = []
        err_records = []
        last_collected_date = None
        daily_stats = []
        print('새로 수집 시작')

    # 시작 ~ 끝 일자를 하루씩 순회
    current = datetime.strptime(start_date, '%Y.%m.%d')
    end = datetime.strptime(end_date, '%Y.%m.%d')

    while current <= end:
        day_str = current.strftime('%Y.%m.%d')

        # 이미 수집 완료한 날짜면 건너뜀 (재시작 시 중복 fetch 차단)
        if last_collected_date and day_str <= last_collected_date:
            print(f"{day_str} — 이미 수집 완료, 건너뜀")
            current += timedelta(days=1)
            continue

        date_ymd = day_str.replace('.', '')  # 'YYYYMMDD' — API datetimeBegin/End용
        started_at = time.time()

        # 하루치 모든 페이지 받기 (count → 페이지 순회)
        articles, total, max_page, page_stats = collect_articles_for_day(date_ymd)

        # 각 기사를 표준 행으로 변환 — 필드 누락 시 err_records에 기록하고 다음 기사로
        day_ok = 0
        day_err = 0
        for article in articles:
            try:
                row = article_to_row(article)
                all_rows.append(row)
                day_ok += 1
            except Exception as exc:
                # newsCode가 있으면 URL 재구성 가능, 없으면 그대로 dict 일부 보존
                err_records.append({
                    'date': day_str,
                    'news_code': article.get('newsCode'),
                    'error': repr(exc),
                })
                day_err += 1

        elapsed = round(time.time() - started_at, 2)

        # 일자별 수집량/페이지 수/소요 시간/오류 수 로그 — 사후 진단용
        daily_stats.append({
            'date': day_str,
            'api_total': total,
            'received': len(articles),
            'parsed_ok': day_ok,
            'parsed_err': day_err,
            'max_page': max_page,
            'elapsed_sec': elapsed,
            'pages': page_stats,
        })

        print(
            f"{day_str} — API 총 {total}건 / 받음 {len(articles)}건 / OK {day_ok}건 / 오류 {day_err}건 "
            f"/ 페이지 {max_page} / {elapsed}초 / 누적 {len(all_rows)}건"
        )

        # 하루치 수집 후 임시 파일에 즉시 저장 (중간에 끊겨도 누적 보존 + 마지막 완료 날짜 기록)
        with temp_path.open('w', encoding='utf-8') as f:
            json.dump({
                'rows': all_rows,
                'err_records': err_records,
                'last_date': day_str,
                'daily_stats': daily_stats,
            }, f, ensure_ascii=False, indent=2)
        last_collected_date = day_str
        current += timedelta(days=1)
        # 다음 날짜로 넘어가기 전 대기 (마지막 날 뒤엔 안 함)
        if current <= end:
            polite_sleep('다음 날짜 전', DAY_PAUSE_RANGE_SEC)

    if not all_rows:
        raise ValueError('수집된 본문 데이터가 없습니다.')

    # 수집한 정보들을 dataframe으로 변환
    df = pd.DataFrame(all_rows)

    # 수집한 기사들 중 중복인 경우 이를 제거 (link 기준)
    df_no_duplicates = df.drop_duplicates(subset=['link']).reset_index(drop=True)

    # 오래된 순부터 수집했으나 혹시 모를 상황을 방지하기 위해 pubdate를 datetime으로 변환 후 정렬
    df_no_duplicates['pubdate'] = pd.to_datetime(df_no_duplicates['pubdate'], errors='coerce')
    df_sorted = df_no_duplicates.sort_values(by='pubdate').reset_index(drop=True)

    # 수집한 정보들을 csv로 저장 (Google Drive에 저장)
    df_sorted.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f'저장 완료: {csv_save_path}')
    print(f'행 수: {len(df)} -> dedup {len(df_no_duplicates)} -> 저장 {len(df_sorted)}')

    # 수집 로그 — 일자별 통계 + 전체 요약
    with stats_save_path.open('w', encoding='utf-8') as f:
        json.dump({
            'press': press,
            'start_date': start_date,
            'end_date': end_date,
            'total_rows': len(df_sorted),
            'err_count': len(err_records),
            'days': daily_stats,
            'err_records': err_records,
        }, f, ensure_ascii=False, indent=2)
    print(f'수집 로그 저장: {stats_save_path}')

    # 정상 완료 시 임시 파일 삭제 — 다음 실행에서 그릇된 재개 방지
    if temp_path.exists():
        temp_path.unlink()

    print(f'본문 수집 완료 — 총 {len(df_sorted)}건 / 오류 {len(err_records)}건')
    return csv_save_path


# 생성된 jobs를 순서대로 실행
# 한 작업이 실패해도 실패 목록에 기록하고 다음 작업으로 넘어감
results = []
failures = []
for index, job in enumerate(jobs, start=1):
    print()
    print(f"[{index}/{len(jobs)}] 작업 실행: {job}")
    try:
        # job 딕셔너리의 press/start_date/end_date를 collect_for_period 인자로 전달
        results.append(collect_for_period(**job))
    except Exception as exc:
        # 한 언론사에서 오류가 나도 전체 작업이 멈추지 않도록 실패 정보만 저장
        failures.append({'job': job, 'error': repr(exc)})
        print(f"작업 실패, 다음 작업으로 넘어감: {exc!r}")
    finally:
        if index < len(jobs):
            # 다음 언론사 job으로 넘어가기 전 대기
            polite_sleep('다음 작업 전', JOB_PAUSE_RANGE_SEC)

# 실패한 작업이 있으면 나중에 다시 돌릴 수 있게 파일로 저장
if failures:
    failures_path = SAVE_DIR / 'KBS_수집실패목록.json'
    with failures_path.open('w', encoding='utf-8') as f:
        json.dump(failures, f, ensure_ascii=False, indent=2)
    print()
    print(f"실패 작업 {len(failures)}개 저장: {failures_path}")

print()
print('전체 작업 완료')
print(f'성공/건너뜀: {len(results)}개, 실패: {len(failures)}개')
for result_path in results:
    print(result_path)